In [1]:
import pandas as pd

In [2]:
import sklearn
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.metrics import root_mean_squared_error
import mlflow

In [3]:
mlflow.set_tracking_uri('sqlite:////workspaces/mlops_zoomcamp/mlflow.db')
mlflow.set_experiment('my_nyc_experiment')

<Experiment: artifact_location='/workspaces/mlops_zoomcamp/02-experiment/mlruns/1', creation_time=1784703584028, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1784703584028, lifecycle_stage='active', name='my_nyc_experiment', tags={}, trace_location=None, workspace='default'>

In [4]:
#!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet
#!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet

In [5]:
def process_data(dv, categorical, numerical, target, data_path, val):
    data = pd.read_parquet(data_path)
    data['duration'] = data['tpep_dropoff_datetime'] - data ['tpep_pickup_datetime']
    data['duration'] = data['duration'].apply(lambda pd: pd.total_seconds()/60)
    data = data[(data['duration'] > 1) & (data ['duration'] <= 60)]
    data[categorical] = data[categorical].astype(str)
    data_dicts = data[categorical + numerical].to_dict(orient='records')
    if val:
        x_train = dv.transform(data_dicts)
    else:
        x_train = dv.fit_transform(data_dicts)
    y_train = data[target].values
    return x_train, y_train

In [6]:
categorical = ['PULocationID', 'DOLocationID']
numerical = ['trip_distance']
target = 'duration'
dv = DictVectorizer()
x_train, y_train = process_data(dv, categorical, numerical, target, "/workspaces/mlops_zoomcamp/01-intro/yellow_tripdata_2023-01.parquet", val=False)
x_val, y_val = process_data(dv, categorical, numerical, target, "/workspaces/mlops_zoomcamp/01-intro/yellow_tripdata_2023-02.parquet", True)

In [7]:
lr = LinearRegression()
lr.fit(x_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [8]:
y_predict = lr.predict(x_train)
root_mean_squared_error(y_train, y_predict)

7.657364639182448

In [9]:
y_predict_val = lr.predict(x_val)
root_mean_squared_error(y_val, y_predict_val)

7.818374028367057

In [10]:
with mlflow.start_run():
    mlflow.set_tag("developer","rajesh")
    mlflow.log_param("train_data","yellow_tripdata_2023-01.parquet")
    mlflow.log_param("test_data","yellow_tripdata_2023-02.parquet")
    alpha = 0.001
    mlflow.log_param("alpha",alpha)
    lasso = Lasso(alpha)
    lasso.fit(x_train, y_train)
    y_pred = lasso.predict(x_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_param("rmse",rmse)

In [7]:
!pip install xgboost
import xgboost as xgb

In [8]:
train = xgb.DMatrix(x_train, label = y_train)
test = xgb.DMatrix(x_val, label = y_val)

In [9]:
!pip install hyperopt
from hyperopt import fmin, hp, STATUS_OK, Trials, tpe
from hyperopt.pyll import scope

/home/codespace/anaconda3/lib/python3.13/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [10]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model","xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            num_boost_round=1000,
            early_stopping_rounds=50,
            dtrain= train,
            evals=[(test,"validation")]
        )
        y_pred = booster.predict(test)
        rmse = root_mean_squared_error(y_pred, y_val)
        mlflow.log_param("rmse",rmse)
    return {"loss":rmse, "status":STATUS_OK}

In [11]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:squarederror',
    'seed': 42
}

In [ ]:
best_result = fmin(fn = objective,space= search_space, max_evals=10, trials=Trials(), algo=tpe.suggest) 

[0]	validation-rmse:9.31588                                                                    
[1]	validation-rmse:8.65954                                                                    
[2]	validation-rmse:8.08738                                                                    
[3]	validation-rmse:7.59198                                                                    
[4]	validation-rmse:7.16399                                                                    
[5]	validation-rmse:6.79593                                                                    
[6]	validation-rmse:6.48013                                                                    
[7]	validation-rmse:6.21136                                                                    
[8]	validation-rmse:5.98288                                                                    
[9]	validation-rmse:5.78863                                                                    
[10]	validation-rmse:5.62483            

In [15]:
best_params = {
'learning_rate':0.10162048792105309,
'max_depth':17,
'min_child_weight':7.961115203896526,
'objective':'reg:squarederror',
'reg_alpha':0.05076080912582581,
'reg_lambda':0.13932991262367653,
'seed':42,
'rmse':4.520377829971649
}

In [17]:
with mlflow.start_run():
    mlflow.set_tag("model","xgboost_best")
    mlflow.log_params(best_params)
    booster = xgb.train(
            params=best_params,
            num_boost_round=1000,
            early_stopping_rounds=50,
            dtrain= train,
            evals=[(test,"validation")]
            )
    y_pred = booster.predict(test)
    rmse = root_mean_squared_error(y_pred, y_val)
    mlflow.log_param("rmse",rmse)

/home/codespace/anaconda3/lib/python3.13/site-packages/xgboost/callback.py:385: UserWarning: [07:28:27] WARNING: /__w/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "rmse" } are not used.

  self.starting_round = model.num_boosted_rounds()


[0]	validation-rmse:9.31588
[1]	validation-rmse:8.65954
[2]	validation-rmse:8.08738
[3]	validation-rmse:7.59198
[4]	validation-rmse:7.16399
[5]	validation-rmse:6.79593
[6]	validation-rmse:6.48013
[7]	validation-rmse:6.21136
[8]	validation-rmse:5.98288
[9]	validation-rmse:5.78863
[10]	validation-rmse:5.62483
[11]	validation-rmse:5.48710
[12]	validation-rmse:5.37138
[13]	validation-rmse:5.27262
[14]	validation-rmse:5.19089
[15]	validation-rmse:5.12158
[16]	validation-rmse:5.06282
[17]	validation-rmse:5.01386
[18]	validation-rmse:4.97181
[19]	validation-rmse:4.93599
[20]	validation-rmse:4.90588
[21]	validation-rmse:4.87953
[22]	validation-rmse:4.85695
[23]	validation-rmse:4.83815
[24]	validation-rmse:4.82195
[25]	validation-rmse:4.80744
[26]	validation-rmse:4.79380
[27]	validation-rmse:4.78215
[28]	validation-rmse:4.77277
[29]	validation-rmse:4.76384
[30]	validation-rmse:4.75605
[31]	validation-rmse:4.74974
[32]	validation-rmse:4.74314
[33]	validation-rmse:4.73767
[34]	validation-rmse:4.7

In [18]:
mlflow.autolog()

2026/07/23 07:39:01 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/07/23 07:39:01 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.
